# LeetCode 4009 — Minimum Possible Maximum Waiting Time bằng Python

- Link: https://leetcode.com/problems/minimum-possible-maximum-waiting-time/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP), Memoization, Mô phỏng, Nén trạng thái

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minMaxWaitingTime(self, demand: List[int], fuel: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có hai máy bơm xăng, mỗi máy có sẵn một lượng xăng. Các xe xếp hàng và **được phép bơm theo thứ tự chỉ số**. Xe `i` được "phép" khi xe `i-1` **bắt đầu** bơm. Mỗi xe chọn một máy còn đủ xăng, chờ máy đó rảnh rồi bơm ngay. Thời gian xe phải chờ = lúc thực sự bắt đầu − lúc được phép.

Đề yêu cầu **hai tầng mục tiêu**: trước hết **phục vụ được nhiều xe nhất** (một tiền tố của dãy xe), rồi trong tất cả cách phục vụ được nhiều xe nhất đó, tìm cách làm **thời gian chờ lớn nhất** (max waiting time) **nhỏ nhất**. Nếu không phục vụ được xe nào, trả `-1`.

Bài này nhìn đáng sợ vì có "thời gian", nhưng thật ra chỉ là một bài **DP trên trạng thái nhỏ**, miễn là bạn biết **nén** trạng thái: thay vì lưu thời gian tuyệt đối, chỉ lưu thời gian rảnh **tương đối** so với thời điểm hiện tại.

Sau notebook này bạn sẽ:

1. Mô phỏng chính xác luật của đề bằng một hàm `simulate` cho một phương án chọn máy cho trước.
2. Viết lời giải vét cạn `O(2^n)` để làm "trọng tài" kiểm chứng.
3. Hiểu cách **nén trạng thái** về `(i, xăng_máy_0, xăng_máy_1, chờ_0, chờ_1)` và viết DP có nhớ (memoization).
4. Xử lý mục tiêu **hai tầng** bằng cách so sánh cặp `(-số_xe_phục_vụ, thời_gian_chờ_lớn_nhất)` theo thứ tự từ điển.
5. Viết thêm hai cách khác (DP tiến theo từng lớp, và chặt nhị phân đáp án) rồi đối chiếu chúng bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại bằng lời dễ hiểu.**

- `demand[i]` = lượng xăng xe `i` cần. `fuel = [f0, f1]` = lượng xăng ban đầu của hai máy.
- Xe `0` được phép tại thời điểm `0`. Xe `i > 0` được phép **đúng lúc xe `i-1` bắt đầu bơm**.
- Khi xe `i` được phép, bạn **phải** chọn một máy còn **ít nhất** `demand[i]` xăng. Nếu cả hai đều đủ, bạn được chọn tuỳ ý (kể cả máy đang bận lâu hơn).
- Xe chờ đến lúc máy đã chọn rảnh rồi bơm **ngay**, máy bận trong `demand[i]` giây, và xăng của máy giảm `demand[i]`.
- Nếu **cả hai** máy đều không đủ xăng cho xe `i`, quá trình **dừng**, các xe sau không được phục vụ.
- **Thời gian chờ** của xe = lúc bắt đầu bơm − lúc được phép.
- Trả về giá trị **nhỏ nhất** của **thời gian chờ lớn nhất**, xét trong số các cách phân máy **phục vụ được nhiều xe nhất**. Không phục vụ được xe nào thì trả `-1`.

**Ví dụ 1:** `demand = [6,8,4,6,5]`, `fuel = [16,13]` → `6`. Đề cho một phương án phục vụ cả 5 xe: máy 0 bơm các xe cần 6, 4, 6; máy 1 bơm các xe cần 8, 5. Xe 2 (cần 4) phải chờ máy 0 xong xe 0 (6 giây) nên chờ 6. Không phương án nào phục vụ đủ 5 xe mà chờ ít hơn.

**Ví dụ 2:** `demand = [10,15]`, `fuel = [12,17]` → `0`. Xe 0 vào máy 0, xe 1 vào máy 1, không ai phải chờ.

**Ví dụ 3:** `demand = [10,5]`, `fuel = [8,8]` → `-1`. Xe 0 cần 10 nhưng mỗi máy chỉ có 8: không phục vụ được xe nào.

**Ràng buộc**

- `1 <= demand.length <= 50`
- `1 <= demand[i] <= 20`
- `fuel.length == 2`
- `1 <= fuel[i] <= 50`

In [1]:
from typing import List

EXAMPLES = [
    ([6, 8, 4, 6, 5], [16, 13], 6),
    ([10, 15], [12, 17], 0),
    ([10, 5], [8, 8], -1),
]
for demand, fuel, expected in EXAMPLES:
    print(f"demand={demand}, fuel={fuel}  ->  kỳ vọng {expected}")

demand=[6, 8, 4, 6, 5], fuel=[16, 13]  ->  kỳ vọng 6
demand=[10, 15], fuel=[12, 17]  ->  kỳ vọng 0
demand=[10, 5], fuel=[8, 8]  ->  kỳ vọng -1


## 3. Kiến thức nền

Ta ôn các công cụ Python nhỏ sẽ dùng, mỗi thứ kèm một ví dụ chạy được.

### 3.1. So sánh tuple (thứ tự từ điển) và `min` trên tuple

Python so sánh hai tuple **từng phần tử từ trái sang phải**: phần tử đầu khác nhau thì quyết định luôn, bằng nhau mới xét phần tử kế tiếp. Điều này rất hợp với mục tiêu **hai tầng** của đề:

- Tầng 1: **nhiều xe nhất** → ta lưu `-số_xe` để "nhỏ hơn" nghĩa là "nhiều xe hơn".
- Tầng 2: **chờ lớn nhất nhỏ nhất** → phần tử thứ hai là `max_wait`.

Chỉ cần `min(...)` trên các tuple dạng `(-số_xe, max_wait)` là đúng ngay thứ tự ưu tiên của đề.

In [2]:
a = (-5, 9)    # phục vụ 5 xe, chờ lớn nhất 9
b = (-4, 0)    # phục vụ 4 xe, chờ lớn nhất 0
c = (-5, 6)    # phục vụ 5 xe, chờ lớn nhất 6
print(a < b)          # True : -5 < -4  (nhiều xe hơn thắng, dù chờ lâu hơn)
print(c < a)          # True : cùng 5 xe, chờ 6 < 9
print(min(a, b, c))   # (-5, 6)

True
True
(-5, 6)


### 3.2. `lru_cache` — nhớ kết quả của hàm đệ quy

`functools.lru_cache(None)` bọc một hàm để lần gọi thứ hai với **cùng tham số** trả về kết quả đã lưu thay vì tính lại. Tham số phải **băm được** (số nguyên, tuple... đều ổn). Đây là cách nhanh nhất để biến "đệ quy vét cạn" thành "DP top-down".

In [3]:
from functools import lru_cache

calls = 0

@lru_cache(None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), "số lần thực sự tính:", calls)   # 31 lần thay vì ~2.7 triệu
print(fib.cache_info())

832040 số lần thực sự tính: 31
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


### 3.3. `max(0, x)` để "kẹp" giá trị âm và dict làm bảng min

Thời gian còn phải chờ của một máy không thể âm: nếu máy đã rảnh từ lâu thì chờ là `0`. Ta dùng `max(0, x)` để **kẹp** (clamp).

Khi DP theo lớp, ta giữ dict `{trạng thái: giá trị nhỏ nhất}` và cập nhật bằng `if key not in d or val < d[key]`.

In [4]:
for x in [5, 0, -3]:
    print(x, "->", max(0, x))

best = {}
for key, val in [(("A",), 7), (("A",), 4), (("B",), 9), (("A",), 6)]:
    if key not in best or val < best[key]:
        best[key] = val
print(best)   # {('A',): 4, ('B',): 9}

5 -> 5
0 -> 0
-3 -> 0
{('A',): 4, ('B',): 9}


## 4. Ý tưởng nền tảng / trực giác

Mỗi xe chỉ có **hai lựa chọn** (máy 0 hoặc máy 1), nên cả quá trình là một dãy lựa chọn nhị phân. Nếu ai đó **cho sẵn** dãy lựa chọn `choices`, việc tính thời gian chờ chỉ là mô phỏng thuần tuý.

Tính tay ví dụ 1 với `choices = [0, 1, 0, 0, 1]` (xe 0→máy 0, xe 1→máy 1, xe 2→máy 0, xe 3→máy 0, xe 4→máy 1):

| xe | được phép (`t`) | máy | máy rảnh lúc | bắt đầu | chờ | xăng máy sau khi bơm |
|---|---|---|---|---|---|---|
| 0 (cần 6) | 0 | 0 | 0 | 0 | 0 | (10, 13) |
| 1 (cần 8) | 0 | 1 | 0 | 0 | 0 | (10, 5) |
| 2 (cần 4) | 0 | 0 | 6 | 6 | 6 | (6, 5) |
| 3 (cần 6) | 6 | 0 | 10 | 10 | 4 | (0, 5) |
| 4 (cần 5) | 10 | 1 | 8 | 10 | 0 | (0, 0) |

Chú ý: "được phép" của xe `i` chính là **lúc bắt đầu** của xe `i-1`, không phải lúc xe `i-1` bơm xong. Hai quy tắc quan trọng:

- Lúc bắt đầu của xe `i` = `max(t_i, rảnh_của_máy_đã_chọn)`.
- Chờ của xe `i` = `bắt_đầu − t_i = max(0, rảnh − t_i)`.

Ta viết đúng hàm đó.

In [5]:
def simulate(demand, fuel, choices, verbose=False):
    """Mô phỏng khi đã biết choices[i] ∈ {0,1}.
    Trả về (số_xe_phục_vụ, chờ_lớn_nhất, tổng_chờ), hoặc None nếu có lựa chọn phi pháp
    (chọn máy thiếu xăng trong khi máy kia đủ)."""
    rem = list(fuel)          # xăng còn lại của 2 máy
    free = [0, 0]             # thời điểm mỗi máy rảnh
    t = 0                     # thời điểm xe hiện tại được phép
    served = max_wait = total_wait = 0
    for i, d in enumerate(demand):
        if rem[0] < d and rem[1] < d:
            break             # cả hai máy đều thiếu -> dừng
        j = choices[i]
        if rem[j] < d:
            return None       # chọn máy thiếu xăng khi máy kia vẫn đủ: không hợp lệ
        start = max(t, free[j])
        wait = start - t
        if verbose:
            print(f"xe {i} (cần {d}): được phép t={t}, máy {j}, bắt đầu {start}, chờ {wait}")
        rem[j] -= d
        free[j] = start + d
        t = start             # xe kế tiếp được phép khi xe này BẮT ĐẦU
        served += 1
        max_wait = max(max_wait, wait)
        total_wait += wait
    return served, max_wait, total_wait

print(simulate([6, 8, 4, 6, 5], [16, 13], [0, 1, 0, 0, 1], verbose=True))

xe 0 (cần 6): được phép t=0, máy 0, bắt đầu 0, chờ 0
xe 1 (cần 8): được phép t=0, máy 1, bắt đầu 0, chờ 0
xe 2 (cần 4): được phép t=0, máy 0, bắt đầu 6, chờ 6
xe 3 (cần 6): được phép t=6, máy 0, bắt đầu 10, chờ 4
xe 4 (cần 5): được phép t=10, máy 1, bắt đầu 10, chờ 0
(5, 6, 10)


## 5. Cách 1 — Brute Force: thử mọi cách chọn máy

**Ý tưởng.** Với mỗi xe, thử cả hai máy (nếu đủ xăng), đệ quy tới xe kế tiếp, **mang theo thời gian tuyệt đối**. Mỗi khi "dừng" (hết xe hoặc cả hai máy thiếu xăng) ta ghi nhận kết quả `(số xe phục vụ, chờ lớn nhất)` và giữ cái tốt nhất theo thứ tự từ điển.

Đây là bản dịch gần như nguyên văn của đề, nên rất đáng tin để làm trọng tài.

In [6]:
def brute_force(demand, fuel):
    n = len(demand)
    best = [(0, 0)]          # lưu (-số_xe, max_wait) tốt nhất; khởi tạo: 0 xe

    def dfs(i, rem0, rem1, free0, free1, t, max_wait):
        # i xe đầu đã được phục vụ
        cand = (-i, max_wait)
        if cand < best[0]:
            best[0] = cand
        if i == n:
            return
        d = demand[i]
        if rem0 >= d:
            start = max(t, free0)
            dfs(i + 1, rem0 - d, rem1, start + d, free1, start, max(max_wait, start - t))
        if rem1 >= d:
            start = max(t, free1)
            dfs(i + 1, rem0, rem1 - d, free0, start + d, start, max(max_wait, start - t))

    dfs(0, fuel[0], fuel[1], 0, 0, 0, 0)
    served, wait = -best[0][0], best[0][1]
    return -1 if served == 0 else wait

for demand, fuel, expected in EXAMPLES:
    print(demand, fuel, "->", brute_force(demand, fuel), "(kỳ vọng", expected, ")")

[6, 8, 4, 6, 5] [16, 13] -> 6 (kỳ vọng 6 )
[10, 15] [12, 17] -> 0 (kỳ vọng 0 )
[10, 5] [8, 8] -> -1 (kỳ vọng -1 )


**Độ phức tạp.** Mỗi xe có tối đa 2 nhánh nên `O(2^n)` thời gian, `O(n)` bộ nhớ đệ quy. Với `n = 50` là không thể. **Nút thắt:** rất nhiều nhánh khác nhau dẫn tới **cùng một tình huống** (cùng xăng còn lại, cùng "ai bận bao lâu nữa"), và ta lại tính lại từ đầu mỗi lần. Chỉ cần nhận ra "tình huống" nào là đủ để quyết định tương lai.

## 6. Bước đột phá — nén trạng thái bằng thời gian tương đối

**Câu hỏi then chốt:** để quyết định những xe còn lại, ta cần biết gì về quá khứ?

1. Xe nào sắp tới: chỉ số `i`.
2. Xăng còn lại của hai máy: `r0`, `r1`.
3. Mỗi máy còn **bận thêm bao lâu nữa so với hiện tại** — nhưng thời gian tuyệt đối `t` thì không cần. Đặt `a = max(0, rảnh_0 − t)` và `b = max(0, rảnh_1 − t)`.

Vì sao `a` là **thời gian chờ** nếu chọn máy 0 ngay bây giờ? Vì `chờ = max(0, rảnh − t)`. Máy rảnh trước `t` thì chờ `0` và việc nó rảnh từ bao giờ không còn ảnh hưởng gì.

**Quy tắc chuyển trạng thái.** Ở trạng thái `(i, r0, r1, a, b)` với `d = demand[i]`:

- Nếu `r0 >= d`, chọn máy 0: chờ `a`; thời điểm mới `t' = t + a`. Máy 0 bận thêm `d` kể từ `t'`, tức `a' = d`. Máy 1 trước còn bận `b` kể từ `t`, nay `t` đã dịch `a`, nên `b' = max(0, b − a)`. Trạng thái mới `(i+1, r0−d, r1, d, max(0, b−a))`.
- Nếu `r1 >= d`, đối xứng: `(i+1, r0, r1−d, max(0, a−b), d)`.

Một chi tiết hay: do mọi xe được phục vụ đều nhận đủ xăng nên `r0 + r1 = fuel0 + fuel1 − tổng demand của i xe đầu`, vì thế `r1` được xác định bởi `i` và `r0`. Số trạng thái thật sự chỉ cỡ `n × F × D²` với `F ≤ 50`, `D ≤ 20`.

Hãy kiểm chứng: **nén** có thật sự làm giảm số trạng thái khác nhau so với dùng thời gian tuyệt đối không?

In [7]:
import random

def count_states(demand, fuel, relative):
    """Đếm số trạng thái phân biệt ở tất cả các lớp khi dùng thời gian tuyệt đối vs tương đối."""
    layer = {(fuel[0], fuel[1], 0, 0, 0)}   # (r0, r1, free0, free1, t) tuyệt đối
    total = 1
    for d in demand:
        nxt = set()
        for r0, r1, f0, f1, t in layer:
            if r0 >= d:
                s = max(t, f0)
                nxt.add((r0 - d, r1, s + d, f1, s))
            if r1 >= d:
                s = max(t, f1)
                nxt.add((r0, r1 - d, f0, s + d, s))
        if not nxt:
            break
        if relative:
            nxt = {(r0, r1, max(0, f0 - t), max(0, f1 - t), 0) for r0, r1, f0, f1, t in nxt}
        layer = nxt
        total += len(layer)
    return total

rng = random.Random(1)
demand = [rng.randint(1, 5) for _ in range(14)]
fuel = [30, 30]
print("demand =", demand)
print("trạng thái (tuyệt đối):", count_states(demand, fuel, relative=False))
print("trạng thái (tương đối):", count_states(demand, fuel, relative=True))

demand = [2, 5, 1, 3, 1, 4, 4, 4, 4, 2, 1, 4, 1, 4]
trạng thái (tuyệt đối): 6335
trạng thái (tương đối): 975


Số trạng thái giảm mạnh. Giờ ta chỉ việc biến đệ quy thành đệ quy có nhớ trên trạng thái tương đối.

## 7. Các cách tiếp theo

Ta trình bày 3 cách nữa, tất cả dựa trên trạng thái tương đối:

- **Cách 2:** DP top-down (memoization) trả về cặp `(-số_xe, max_wait)`.
- **Cách 3:** DP **bottom-up theo lớp**: với mỗi `i` giữ dict `{trạng thái: max_wait nhỏ nhất đạt được}`.
- **Cách 4:** Tách hai tầng mục tiêu: tính trước `k` = số xe tối đa, rồi **chặt nhị phân** `W` và kiểm tra "có phương án phục vụ `k` xe mà mọi lần chờ `≤ W` không?".

### 7.1. Cách 2 — DP top-down

**Giải thích từng đoạn.**

1. `go(i, r0, r1, a, b)` trả về giá trị tốt nhất của **phần còn lại** dưới dạng `(-số_xe_thêm, max_wait_phần_còn_lại)`.
2. Nếu `i == n`: hết xe, trả `(0, 0)`.
3. Với mỗi máy đủ xăng, ta tính `(sub_neg - 1, max(chờ_hiện_tại, sub_wait))` — thêm một xe được phục vụ, chờ lớn nhất là lớn nhất giữa lần chờ hiện tại và phần sau.
4. Nếu không máy nào đủ xăng: quá trình dừng, trả `(0, 0)`.
5. Lấy `min` các lựa chọn theo thứ tự từ điển.

In [8]:
def min_max_wait_memo(demand, fuel):
    n = len(demand)

    @lru_cache(None)
    def go(i, r0, r1, a, b):
        if i == n:
            return (0, 0)
        d = demand[i]
        options = []
        if r0 >= d:                                   # chọn máy 0: chờ a
            neg, w = go(i + 1, r0 - d, r1, d, max(0, b - a))
            options.append((neg - 1, max(a, w)))
        if r1 >= d:                                   # chọn máy 1: chờ b
            neg, w = go(i + 1, r0, r1 - d, max(0, a - b), d)
            options.append((neg - 1, max(b, w)))
        return min(options) if options else (0, 0)

    neg, w = go(0, fuel[0], fuel[1], 0, 0)
    return -1 if neg == 0 else w

for demand, fuel, expected in EXAMPLES:
    assert min_max_wait_memo(demand, fuel) == expected
    print(demand, fuel, "->", min_max_wait_memo(demand, fuel))

[6, 8, 4, 6, 5] [16, 13] -> 6
[10, 15] [12, 17] -> 0
[10, 5] [8, 8] -> -1


**Độ phức tạp.** Số trạng thái `O(n · F · D²)` (với `F` ≈ xăng tối đa một máy, `D` ≈ demand tối đa), mỗi trạng thái 2 chuyển → `O(n · F · D²)` thời gian và bộ nhớ. Với `50 · 51 · 21 · 21 ≈ 1.1 triệu` trạng thái là lý thuyết xấu nhất; thực tế ít hơn rất nhiều.

### 7.2. Cách 3 — DP bottom-up theo lớp

**Ý tưởng.** Cùng bảng trạng thái nhưng đi **tiến** từ `i = 0`. Lớp `i` là dict `{(r0, r1, a, b): max_wait nhỏ nhất của các cách đến đây}`. Khi lớp kế tiếp rỗng nghĩa là **không ai** phục vụ thêm được nữa: số xe tối đa = số lớp khác rỗng cuối cùng, và đáp án là `min` các giá trị ở lớp đó. Cách này không cần đệ quy nên không lo giới hạn độ sâu.

Điểm tinh tế: ta lưu **max_wait của quá khứ** (đến hiện tại), và `max` kết hợp quá khứ với lần chờ hiện tại. Vì `max` là phép toán **đơn điệu**, giữ lại giá trị nhỏ nhất cho mỗi trạng thái là đủ (nếu hai quá khứ cùng dẫn tới một trạng thái, quá khứ có max_wait nhỏ hơn luôn không tệ hơn).

In [9]:
def min_max_wait_layers(demand, fuel):
    layer = {(fuel[0], fuel[1], 0, 0): 0}     # trạng thái -> max_wait nhỏ nhất
    served = 0
    for i, d in enumerate(demand):
        nxt = {}
        for (r0, r1, a, b), w in layer.items():
            if r0 >= d:
                key, val = (r0 - d, r1, d, max(0, b - a)), max(w, a)
                if key not in nxt or val < nxt[key]:
                    nxt[key] = val
            if r1 >= d:
                key, val = (r0, r1 - d, max(0, a - b), d), max(w, b)
                if key not in nxt or val < nxt[key]:
                    nxt[key] = val
        if not nxt:
            break                              # không ai phục vụ được xe i
        layer, served = nxt, i + 1
    return -1 if served == 0 else min(layer.values())

for demand, fuel, expected in EXAMPLES:
    assert min_max_wait_layers(demand, fuel) == expected
print("Cách 3 khớp 3 ví dụ")

Cách 3 khớp 3 ví dụ


**Độ phức tạp.** Giống cách 2: `O(n · F · D²)` thời gian; bộ nhớ chỉ cần **hai lớp** nên `O(F · D²)`.

### 7.3. Cách 4 — Tách hai tầng: số xe tối đa, rồi chặt nhị phân thời gian chờ

**Ý tưởng.**

1. **Tầng 1** không liên quan thời gian: xe `0..k-1` phục vụ được iff chia được tập `demand[0..k-1]` vào hai "túi" sức chứa `fuel[0]`, `fuel[1]` — chỉ cần tập `{(r0, r1)}` đạt được. `k` = tiền tố dài nhất còn đạt được.
2. **Tầng 2**: với `k` cố định, hàm `feasible(W)` = "có cách phục vụ đủ `k` xe mà mọi lần chờ `≤ W` không?". `feasible` **đơn điệu** theo `W` (cho phép chờ nhiều hơn chỉ làm dễ hơn), nên chặt nhị phân tìm `W` nhỏ nhất.
3. Cận trên của `W`: `sum(demand)` luôn đủ vì không ai chờ quá tổng thời gian bơm của các xe trước.

In [10]:
def max_served(demand, fuel):
    states = {(fuel[0], fuel[1])}
    k = 0
    for d in demand:
        nxt = set()
        for r0, r1 in states:
            if r0 >= d:
                nxt.add((r0 - d, r1))
            if r1 >= d:
                nxt.add((r0, r1 - d))
        if not nxt:
            break
        states, k = nxt, k + 1
    return k

def feasible(demand, fuel, k, W):
    layer = {(fuel[0], fuel[1], 0, 0)}
    for i in range(k):
        d = demand[i]
        nxt = set()
        for r0, r1, a, b in layer:
            if r0 >= d and a <= W:
                nxt.add((r0 - d, r1, d, max(0, b - a)))
            if r1 >= d and b <= W:
                nxt.add((r0, r1 - d, max(0, a - b), d))
        layer = nxt
        if not layer:
            return False
    return True

def min_max_wait_bsearch(demand, fuel):
    k = max_served(demand, fuel)
    if k == 0:
        return -1
    lo, hi = 0, sum(demand)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(demand, fuel, k, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for demand, fuel, expected in EXAMPLES:
    assert min_max_wait_bsearch(demand, fuel) == expected
print("Cách 4 khớp 3 ví dụ; xe tối đa ở ví dụ 1:", max_served(*EXAMPLES[0][:2]))

Cách 4 khớp 3 ví dụ; xe tối đa ở ví dụ 1: 5


**Độ phức tạp.** Mỗi lần `feasible` tốn `O(k · F · D²)`, nhân với `log(sum(demand))` ≈ 10 lần → `O(n · F · D² · log S)`. Chậm hơn cách 2/3 nhưng cho ta một góc nhìn: bài toán 2 tầng có thể **tách** thành "đếm" rồi "tìm ngưỡng".

## 8. TRACE từng bước

Ta theo dõi cách 3 (DP theo lớp) trên ví dụ 1 để thấy dict trạng thái lớn dần/nhỏ dần. Mỗi trạng thái là `(r0, r1, a, b)` → `max_wait` quá khứ.

Ý nghĩa cột: `r0, r1` = xăng còn lại; `a, b` = máy 0/1 còn bận bao lâu **tính từ lúc xe sắp tới được phép**.

| lớp i | ý nghĩa |
|---|---|
| 0 | chưa phục vụ xe nào: `(16, 13, 0, 0) → 0` |
| 1 | xe 0 (cần 6) vào máy 0 hoặc máy 1 |
| 2... | mỗi lớp mở rộng 2 nhánh rồi gộp các trạng thái trùng bằng min |

In [11]:
def trace_layers(demand, fuel):
    layer = {(fuel[0], fuel[1], 0, 0): 0}
    print("lớp 0:", layer)
    for i, d in enumerate(demand):
        nxt = {}
        for (r0, r1, a, b), w in layer.items():
            if r0 >= d:
                key, val = (r0 - d, r1, d, max(0, b - a)), max(w, a)
                if key not in nxt or val < nxt[key]:
                    nxt[key] = val
            if r1 >= d:
                key, val = (r0, r1 - d, max(0, a - b), d), max(w, b)
                if key not in nxt or val < nxt[key]:
                    nxt[key] = val
        if not nxt:
            print(f"xe {i} (cần {d}): không máy nào đủ xăng -> DỪNG, đã phục vụ {i} xe")
            break
        layer = nxt
        print(f"\nsau xe {i} (cần {d}): {len(layer)} trạng thái")
        for key, val in sorted(layer.items()):
            print("   ", key, "-> max_wait", val)
    return min(layer.values())

print("\nĐáp án:", trace_layers([6, 8, 4, 6, 5], [16, 13]))

lớp 0: {(16, 13, 0, 0): 0}

sau xe 0 (cần 6): 2 trạng thái
    (10, 13, 6, 0) -> max_wait 0
    (16, 7, 0, 6) -> max_wait 0

sau xe 1 (cần 8): 3 trạng thái
    (2, 13, 8, 0) -> max_wait 6
    (8, 7, 8, 6) -> max_wait 0
    (10, 5, 6, 8) -> max_wait 0

sau xe 2 (cần 4): 5 trạng thái
    (2, 9, 8, 4) -> max_wait 6
    (4, 7, 4, 0) -> max_wait 8
    (6, 5, 4, 2) -> max_wait 6
    (8, 3, 2, 4) -> max_wait 6
    (10, 1, 0, 4) -> max_wait 8

sau xe 3 (cần 6): 5 trạng thái
    (0, 5, 6, 0) -> max_wait 6
    (2, 3, 4, 6) -> max_wait 6
    (2, 3, 6, 2) -> max_wait 6
    (4, 1, 4, 6) -> max_wait 8
    (4, 1, 6, 4) -> max_wait 8

sau xe 4 (cần 5): 1 trạng thái
    (0, 0, 6, 5) -> max_wait 6

Đáp án: 6


Quan sát lớp cuối: chỉ còn các trạng thái mà cả 5 xe đã được phục vụ; ta lấy `min` các `max_wait` → `6`. Trace thứ hai cho ví dụ 3 (dừng ngay từ xe 0):

In [12]:
d = 10
print("xe 0 cần", d, "; xăng hai máy: 8 và 8 -> cả hai đều thiếu -> dừng, phục vụ 0 xe -> trả -1")
print(min_max_wait_layers([10, 5], [8, 8]))

xe 0 cần 10 ; xăng hai máy: 8 và 8 -> cả hai đều thiếu -> dừng, phục vụ 0 xe -> trả -1
-1


## 9. Code nộp LeetCode

Ta nộp **cách 2** (top-down) vì ngắn, dễ đọc và đủ nhanh với `n ≤ 50`, `fuel ≤ 50`, `demand ≤ 20`.

In [13]:
from typing import List
from functools import lru_cache

class Solution:
    def minMaxWaitingTime(self, demand: List[int], fuel: List[int]) -> int:
        n = len(demand)

        @lru_cache(None)
        def go(i: int, r0: int, r1: int, a: int, b: int):
            # (-số xe phục vụ thêm, chờ lớn nhất trong phần còn lại)
            if i == n:
                return (0, 0)
            d = demand[i]
            best = None
            if r0 >= d:
                neg, w = go(i + 1, r0 - d, r1, d, max(0, b - a))
                cand = (neg - 1, max(a, w))
                best = cand if best is None or cand < best else best
            if r1 >= d:
                neg, w = go(i + 1, r0, r1 - d, max(0, a - b), d)
                cand = (neg - 1, max(b, w))
                best = cand if best is None or cand < best else best
            return best if best is not None else (0, 0)

        neg, w = go(0, fuel[0], fuel[1], 0, 0)
        return -1 if neg == 0 else w

for demand, fuel, expected in EXAMPLES:
    assert Solution().minMaxWaitingTime(demand, fuel) == expected
print("Code nộp khớp 3 ví dụ")

Code nộp khớp 3 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Trạng thái đủ thông tin.** Tương lai (các xe `i, i+1, ...`) chỉ phụ thuộc: `i`, xăng còn lại `r0, r1`, và thời gian còn phải chờ nếu chọn mỗi máy ngay bây giờ (`a`, `b`). Thời gian tuyệt đối không xuất hiện trong bất kỳ công thức nào ngoài hiệu `rảnh − t`, mà hiệu đó chính là `a`/`b` (kẹp về `0`).

**(b) Chuyển trạng thái đúng.** Chọn máy 0: chờ = `a`, `t' = t + a`. Máy 0 rảnh lúc `t' + d` tức `a' = d`. Máy 1 rảnh lúc `t + b` (nếu còn bận) nên `b' = max(0, t + b − t') = max(0, b − a)`.

**(c) Hai tầng mục tiêu hợp với thứ tự từ điển.** Giá trị của một nhánh là cặp `(−số xe, max_wait)`. Phép "thêm một xe, `max` với lần chờ hiện tại" là **đơn điệu** theo thứ tự từ điển, nên nguyên lý tối ưu con (optimal substructure) đúng: phương án tối ưu của trạng thái gốc dùng phương án tối ưu của trạng thái con.

**(d) Dừng đúng luật.** Chỉ khi **cả hai** máy thiếu xăng ta mới dừng (`(0, 0)`); khi một máy đủ, ta bắt buộc phải phục vụ, không được "bỏ qua" xe — nên mọi nhánh đều là một tiền tố phục vụ liên tục.

**(e) -1.** Nếu xe 0 không phục vụ được thì `neg == 0` → trả `-1`. Nếu phục vụ được thì số xe ≥ 1 nên `neg < 0`, tránh nhầm giữa "0 xe" và "chờ 0".

## 11. Phân tích Time/Space

Gọi `n = len(demand)`, `F = max(fuel)` (≤ 50), `D = max(demand)` (≤ 20).

| Cách | Time | Space |
|---|---|---|
| Brute force | `O(2^n)` | `O(n)` |
| DP top-down (nộp) | `O(n · F · D²)` | `O(n · F · D²)` |
| DP theo lớp | `O(n · F · D²)` | `O(F · D²)` |
| Chặt nhị phân | `O(n · F · D² · log S)` với `S = sum(demand)` | `O(F · D²)` |

Vì sao `D²`: sau mỗi bước, `a` hoặc `b` bằng đúng `d ≤ D` còn giá trị kia chỉ có thể là một giá trị `≤ D` do trước đó (nó chỉ giảm). Vì `r1` được xác định từ `i` và `r0`, số trạng thái thực tế còn nhỏ hơn một bậc so với đếm thô. Hãy đo số trạng thái thực tế trên input lớn:

In [14]:
import time

def states_visited(demand, fuel):
    n = len(demand)
    @lru_cache(None)
    def go(i, r0, r1, a, b):
        if i == n:
            return (0, 0)
        d = demand[i]
        options = []
        if r0 >= d:
            neg, w = go(i + 1, r0 - d, r1, d, max(0, b - a)); options.append((neg - 1, max(a, w)))
        if r1 >= d:
            neg, w = go(i + 1, r0, r1 - d, max(0, a - b), d); options.append((neg - 1, max(b, w)))
        return min(options) if options else (0, 0)
    go(0, fuel[0], fuel[1], 0, 0)
    return go.cache_info().currsize

rng = random.Random(7)
worst = 0
t0 = time.time()
for _ in range(30):
    demand = [rng.randint(1, 20) for _ in range(50)]
    worst = max(worst, states_visited(demand, [50, 50]))
print("số trạng thái lớn nhất gặp được (n=50, fuel 50/50, demand 1..20):", worst)
demand = [rng.randint(1, 3) for _ in range(50)]
t1 = time.time()
print("demand nhỏ (1..3), n=50:", states_visited(demand, [50, 50]), "trạng thái")
print("thời gian:", round(time.time() - t0, 2), "giây")

số trạng thái lớn nhất gặp được (n=50, fuel 50/50, demand 1..20): 1354
demand nhỏ (1..3), n=50: 6073 trạng thái
thời gian: 0.02 giây


## 12. Bộ test

Gồm: ví dụ đề, edge case, một **phản ví dụ** cho chiến lược tham lam, và đối chiếu 4 cách với nhau và với brute force trên input ngẫu nhiên có seed.

In [15]:
# Ví dụ đề + đối chiếu mọi cách
for demand, fuel, expected in EXAMPLES:
    for fn in (brute_force, min_max_wait_memo, min_max_wait_layers, min_max_wait_bsearch):
        assert fn(demand, fuel) == expected, (fn.__name__, demand, fuel)
    assert Solution().minMaxWaitingTime(demand, fuel) == expected
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [16]:
# Edge case
S = Solution()
assert S.minMaxWaitingTime([1], [1, 1]) == 0            # một xe, đủ xăng
assert S.minMaxWaitingTime([5], [4, 4]) == -1           # một xe, thiếu xăng cả hai
assert S.minMaxWaitingTime([3, 3, 3], [6, 3]) == 3      # vừa khít xăng, buộc phải chờ
assert S.minMaxWaitingTime([2, 2, 2, 2], [4, 4]) == 2   # 2 xe mỗi máy
assert S.minMaxWaitingTime([1] * 50, [50, 50]) == min_max_wait_layers([1] * 50, [50, 50])
assert S.minMaxWaitingTime([20, 20, 20], [20, 20]) == 0  # chỉ 2 xe được phục vụ, xe 2 hết xăng
print("Edge case: OK")

Edge case: OK


Một lưu ý ở dòng `[20, 20, 20]` với `fuel = [20, 20]`: chỉ phục vụ được **2 xe** (mỗi xe một máy, chờ `0`), xe 2 không còn xăng nên quá trình dừng. Đáp án là `0`, **không phải** `20`. Bài học: "chờ lớn nhất" chỉ tính trên các xe **được phục vụ**.

**Phản ví dụ cho tham lam.** Chiến lược "luôn chọn máy cho chờ ít nhất (đủ xăng)" nhìn hợp lý nhưng có thể **hết xăng sớm** hoặc đẩy chờ lên về sau. Ta tìm một ví dụ cụ thể bằng tìm kiếm ngẫu nhiên có seed.

In [17]:
def greedy_least_wait(demand, fuel):
    rem, free, t = list(fuel), [0, 0], 0
    served = max_wait = 0
    for d in demand:
        ok = [j for j in (0, 1) if rem[j] >= d]
        if not ok:
            break
        j = min(ok, key=lambda x: max(t, free[x]))   # máy cho bắt đầu sớm nhất
        start = max(t, free[j])
        max_wait = max(max_wait, start - t)
        rem[j] -= d; free[j] = start + d; t = start; served += 1
    return -1 if served == 0 else (served, max_wait)

def true_pair(demand, fuel):
    # (số xe tối đa, chờ nhỏ nhất) từ cách 4
    k = max_served(demand, fuel)
    return None if k == 0 else (k, min_max_wait_bsearch(demand, fuel))

rng = random.Random(2024)
found = None
for _ in range(20000):
    n = rng.randint(2, 6)
    demand = [rng.randint(1, 6) for _ in range(n)]
    fuel = [rng.randint(1, 12), rng.randint(1, 12)]
    g = greedy_least_wait(demand, fuel)
    tp = true_pair(demand, fuel)
    if tp is not None and g != tp:
        found = (demand, fuel, g, tp)
        break
assert found is not None
print("demand =", found[0], "fuel =", found[1])
print("tham lam (số xe, chờ):", found[2], "  đúng:", found[3])

demand = [2, 6, 5, 3, 2] fuel = [12, 7]
tham lam (số xe, chờ): (5, 5)   đúng: (5, 4)


In [18]:
# Đối chiếu nhiều input ngẫu nhiên (seed cố định)
rng = random.Random(12345)
for it in range(1500):
    n = rng.randint(1, 10)
    demand = [rng.randint(1, 8) for _ in range(n)]
    fuel = [rng.randint(1, 20), rng.randint(1, 20)]
    ref = brute_force(demand, fuel)
    assert min_max_wait_memo(demand, fuel) == ref, (demand, fuel)
    assert min_max_wait_layers(demand, fuel) == ref, (demand, fuel)
    assert min_max_wait_bsearch(demand, fuel) == ref, (demand, fuel)
    assert Solution().minMaxWaitingTime(demand, fuel) == ref, (demand, fuel)
print("1500 input ngẫu nhiên: cả 4 cách khớp brute force")

# Input lớn: chỉ so sánh các cách DP với nhau
rng = random.Random(99)
for it in range(40):
    n = rng.randint(30, 50)
    demand = [rng.randint(1, 20) for _ in range(n)]
    fuel = [rng.randint(20, 50), rng.randint(20, 50)]
    a1 = Solution().minMaxWaitingTime(demand, fuel)
    assert a1 == min_max_wait_layers(demand, fuel) == min_max_wait_bsearch(demand, fuel)
print("40 input lớn: các cách DP khớp nhau")

1500 input ngẫu nhiên: cả 4 cách khớp brute force


40 input lớn: các cách DP khớp nhau


In [19]:
# Kiểm tra đáp án khớp với việc thử MỌI dãy lựa chọn bằng simulate
from itertools import product

def by_enumeration(demand, fuel):
    best = None
    for choices in product((0, 1), repeat=len(demand)):
        res = simulate(demand, fuel, choices)
        if res is None:
            continue
        cand = (-res[0], res[1])
        if best is None or cand < best:
            best = cand
    return -1 if best[0] == 0 else best[1]

rng = random.Random(5)
for _ in range(400):
    n = rng.randint(1, 9)
    demand = [rng.randint(1, 7) for _ in range(n)]
    fuel = [rng.randint(1, 18), rng.randint(1, 18)]
    assert by_enumeration(demand, fuel) == Solution().minMaxWaitingTime(demand, fuel), (demand, fuel)
print("simulate + liệt kê mọi dãy lựa chọn: khớp")

simulate + liệt kê mọi dãy lựa chọn: khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ tối ưu thời gian chờ, quên ưu tiên số xe phục vụ

Đề nói "trong các cách **phục vụ nhiều xe nhất**". Nếu chỉ lấy `min` của `max_wait` trên mọi nhánh thì nhánh "dừng sớm với chờ 0" luôn thắng.

In [20]:
def wrong_ignore_count(demand, fuel):
    n = len(demand)
    @lru_cache(None)
    def go(i, r0, r1, a, b):
        if i == n:
            return 0
        d = demand[i]
        opts = [0]                      # SAI: cho phép "dừng" tại đây tuỳ ý
        if r0 >= d: opts.append(max(a, go(i + 1, r0 - d, r1, d, max(0, b - a))))
        if r1 >= d: opts.append(max(b, go(i + 1, r0, r1 - d, max(0, a - b), d)))
        return min(opts)
    return go(0, fuel[0], fuel[1], 0, 0)

demand, fuel = [6, 8, 4, 6, 5], [16, 13]
print("sai :", wrong_ignore_count(demand, fuel))          # 0
print("đúng:", Solution().minMaxWaitingTime(demand, fuel))  # 6

sai : 0
đúng: 6


**Vì sao sai:** dừng sớm là **không được phép** khi vẫn còn máy đủ xăng. **Cách đúng:** so sánh theo cặp `(-số_xe, max_wait)` và chỉ dừng khi hết xe hoặc cả hai máy thiếu xăng.

### Lỗi 2 — Hiểu nhầm "được phép" là lúc xe trước **bơm xong**

Đề nói xe `i` được phép **đúng lúc xe `i-1` bắt đầu**. Nếu tính lúc xe trước xong, thời gian chờ luôn bằng 0 vì ta chờ cho tới khi máy rảnh.

In [21]:
def simulate_wrong_finish(demand, fuel, choices):
    rem, free, t = list(fuel), [0, 0], 0
    max_wait = 0
    for i, d in enumerate(demand):
        j = choices[i]
        if rem[j] < d: break
        start = max(t, free[j])
        max_wait = max(max_wait, start - t)
        rem[j] -= d; free[j] = start + d
        t = start + d                    # SAI: lấy lúc bơm xong
    return max_wait

choices = [0, 1, 0, 0, 1]
print("sai :", simulate_wrong_finish([6, 8, 4, 6, 5], [16, 13], choices))   # không phản ánh đúng
print("đúng:", simulate([6, 8, 4, 6, 5], [16, 13], choices)[1])             # 6

sai : 0
đúng: 6


**Vì sao sai:** với `t = start + d`, xe sau luôn "được phép" khi xe trước đã xong, nên hầu như không ai phải chờ. **Cách đúng:** `t = start` (lúc **bắt đầu**).

### Lỗi 3 — Không kẹp `max(0, ...)` khi chuyển trạng thái tương đối

Nếu viết `b' = b - a` mà không kẹp, giá trị có thể **âm** (máy đã rảnh từ lâu). Sau đó giá trị âm bị dùng như "thời gian chờ" hoặc bị trừ tiếp, làm lệch kết quả.

In [22]:
def wrong_no_clamp(demand, fuel):
    n = len(demand)
    @lru_cache(None)
    def go(i, r0, r1, a, b):
        if i == n:
            return (0, 0)
        d = demand[i]
        o = []
        if r0 >= d:
            neg, w = go(i + 1, r0 - d, r1, d, b - a); o.append((neg - 1, max(a, w)))      # SAI: không kẹp
        if r1 >= d:
            neg, w = go(i + 1, r0, r1 - d, a - b, d); o.append((neg - 1, max(b, w)))      # SAI: không kẹp
        return min(o) if o else (0, 0)
    neg, w = go(0, fuel[0], fuel[1], 0, 0)
    return -1 if neg == 0 else w

rng = random.Random(11)
for _ in range(5000):
    n = rng.randint(3, 8)
    demand = [rng.randint(1, 6) for _ in range(n)]
    fuel = [rng.randint(5, 20), rng.randint(5, 20)]
    if wrong_no_clamp(demand, fuel) != Solution().minMaxWaitingTime(demand, fuel):
        print("demand =", demand, "fuel =", fuel)
        print("sai :", wrong_no_clamp(demand, fuel))
        print("đúng:", Solution().minMaxWaitingTime(demand, fuel))
        break
else:
    raise AssertionError("không tìm được phản ví dụ")

demand = [5, 2, 1, 4, 3, 2, 1, 5] fuel = [6, 17]
sai : 5
đúng: 4


**Vì sao sai:** `a` hay `b` âm nghĩa là "máy đã rảnh trước `t`" nhưng khi đem dùng làm thời gian chờ, nó có thể kéo `max` xuống hoặc làm các phép trừ về sau sai. **Cách đúng:** luôn kẹp `max(0, ·)` — thời gian "chờ nếu chọn máy này" không bao giờ âm, và nhờ vậy các trạng thái tương đương được gộp.

### Lỗi 4 — Cho phép chọn máy thiếu xăng

Chỉ được chọn máy có `rem >= demand[i]`. Quên điều kiện này sẽ cho kết quả nhỏ hơn đáp án thật vì ta "bơm" được từ máy cạn.

In [23]:
def wrong_no_fuel_check(demand, fuel):
    n = len(demand)
    @lru_cache(None)
    def go(i, r0, r1, a, b):
        if i == n:
            return (0, 0)
        d = demand[i]
        o = []
        neg, w = go(i + 1, r0 - d, r1, d, max(0, b - a)); o.append((neg - 1, max(a, w)))   # SAI: bỏ if r0 >= d
        neg, w = go(i + 1, r0, r1 - d, max(0, a - b), d); o.append((neg - 1, max(b, w)))
        return min(o)
    neg, w = go(0, fuel[0], fuel[1], 0, 0)
    return -neg, w

print("sai  (số xe, chờ):", wrong_no_fuel_check([10, 5], [8, 8]))
print("đúng (số xe, chờ):", (0, Solution().minMaxWaitingTime([10, 5], [8, 8])))

sai  (số xe, chờ): (2, 0)
đúng (số xe, chờ): (0, -1)


## 14. Mẹo & nhận dạng pattern

- **"Hai mục tiêu theo thứ tự ưu tiên"** → lưu cặp trong tuple, `min` theo thứ tự từ điển (nhớ đổi dấu cái cần **lớn nhất**).
- **Có thời gian trong mô phỏng** → hãy hỏi "tương lai có thật sự cần thời gian tuyệt đối không?" Thường chỉ cần **chênh lệch** (tương đối). Đây là kỹ thuật nén trạng thái rất phổ biến (lịch trình, hàng đợi, đồng hồ).
- **Ràng buộc nhỏ (≤ 50, ≤ 20)** là tín hiệu cho DP nhiều chiều dù đề dài và lắt léo.
- **`max` của quá khứ và hiện tại** là phép đơn điệu, nên chỉ cần giữ giá trị nhỏ nhất cho từng trạng thái (DP min-max).
- **Mục tiêu min-max** thường có cách thứ hai: chặt nhị phân đáp án + kiểm tra khả thi (cách 4). Bài này không cần nhưng nên biết.
- Luôn viết brute force + `simulate` trước: chúng giúp bạn tin vào đề và bắt lỗi hiểu sai luật.

## 15. Biến thể

1. **Tổng thời gian chờ nhỏ nhất** thay vì chờ lớn nhất: giá trị trở thành tổng cộng (cộng thay vì `max`), vẫn DP trên cùng trạng thái.
2. **`k` máy bơm** thay vì 2: trạng thái thêm chiều nên chỉ làm được khi `k` nhỏ.
3. **Xe được phép khi xe trước bơm xong** (không phải bắt đầu): chờ luôn 0 → bài trở thành thuần "chia xăng".
4. **Cho phép xe bỏ qua nếu không máy nào đủ xăng** (không dừng quá trình): bài trở thành chọn tập con phục vụ.

Ta cài thử biến thể 1 (tổng chờ nhỏ nhất) để thấy chỉ cần đổi phép `max` thành `+`.

In [24]:
def min_total_wait(demand, fuel):
    layer = {(fuel[0], fuel[1], 0, 0): 0}
    served = 0
    for i, d in enumerate(demand):
        nxt = {}
        for (r0, r1, a, b), w in layer.items():
            if r0 >= d:
                key, val = (r0 - d, r1, d, max(0, b - a)), w + a
                if key not in nxt or val < nxt[key]:
                    nxt[key] = val
            if r1 >= d:
                key, val = (r0, r1 - d, max(0, a - b), d), w + b
                if key not in nxt or val < nxt[key]:
                    nxt[key] = val
        if not nxt:
            break
        layer, served = nxt, i + 1
    return -1 if served == 0 else min(layer.values())

def min_total_wait_enum(demand, fuel):
    best = None
    for choices in product((0, 1), repeat=len(demand)):
        res = simulate(demand, fuel, choices)
        if res is None:
            continue
        cand = (-res[0], res[2])
        if best is None or cand < best:
            best = cand
    return -1 if best[0] == 0 else best[1]

rng = random.Random(77)
for _ in range(300):
    n = rng.randint(1, 8)
    demand = [rng.randint(1, 6) for _ in range(n)]
    fuel = [rng.randint(1, 15), rng.randint(1, 15)]
    assert min_total_wait(demand, fuel) == min_total_wait_enum(demand, fuel)
print("Biến thể tổng chờ nhỏ nhất: khớp liệt kê;", min_total_wait([6, 8, 4, 6, 5], [16, 13]))

Biến thể tổng chờ nhỏ nhất: khớp liệt kê; 10


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi dãy chọn máy, mô phỏng thời gian tuyệt đối | `O(2^n)` | `O(n)` | trọng tài, `n ≤ 20` |
| 2. DP top-down (nộp) | đệ quy nhớ trên `(i, r0, r1, a, b)`, trả `(-số_xe, max_wait)` | `O(n·F·D²)` | `O(n·F·D²)` | thi đấu, ngắn gọn |
| 3. DP theo lớp | dict trạng thái → max_wait nhỏ nhất, đi tiến | `O(n·F·D²)` | `O(F·D²)` | muốn tránh đệ quy, tiết kiệm bộ nhớ |
| 4. Chặt nhị phân | đếm `k` rồi tìm ngưỡng `W` nhỏ nhất | `O(n·F·D²·log S)` | `O(F·D²)` | khi cần tách mục tiêu hai tầng |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `simulate([10, 15], [12, 17], [0, 1])` trả về gì? (Gợi ý: ví dụ 2 của đề.)

**Bài 2 (tính tay).** `demand = [3, 3, 3]`, `fuel = [6, 3]`. Phục vụ được mấy xe, và đáp án là bao nhiêu? Hãy vẽ bảng như mục 4.

**Bài 3 (điền TODO).** Hoàn thiện hàm `step(state, d)` trả về danh sách các `(trạng_thái_mới, chờ)` khi xử lý xe cần `d`, với `state = (r0, r1, a, b)`.

**Bài 4 (tự viết lại).** Viết `my_solve(demand, fuel)` bằng DP theo lớp (không dùng đệ quy), dùng `step` của bài 3.

**Bài 5 (biến thể / mini challenge).** Đề phiên bản "xe được phép **khi xe trước bơm xong**" (thay vì bắt đầu). Viết `solve_finish(demand, fuel)` trả về **số xe tối đa** có thể phục vụ (không còn chuyện chờ nữa). Kiểm tra bằng vét cạn.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def step(state, d):
    r0, r1, a, b = state
    out = []
    # TODO 1: nếu r0 >= d -> thêm ((r0 - d, r1, d, max(0, b - a)), a)
    # TODO 2: nếu r1 >= d -> thêm ((r0, r1 - d, max(0, a - b), d), b)
    return out

# Bài 4 — TODO
def my_solve(demand, fuel):
    # TODO: layer = {(fuel[0], fuel[1], 0, 0): 0}; lặp qua các xe, dùng step, giữ max_wait nhỏ nhất
    pass

# Bài 5 — TODO
def solve_finish(demand, fuel):
    # TODO: chỉ cần tập {(r0, r1)} đạt được; trả về số lớp khác rỗng
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: xe 0 cần 10 ≤ 12 nên vào máy 0; xe 1 cần 15 ≤ 17 nên vào máy 1.
- Mức 2: xe 1 được phép lúc `0`, máy 1 rảnh từ `0`.
- Mức 3: kết quả `(2, 0, 0)` = 2 xe, chờ lớn nhất 0, tổng chờ 0.

**Bài 2.**
- Mức 1: tổng demand = 9 = tổng xăng, nên một máy phục vụ 2 xe, một máy 1 xe.
- Mức 2: mọi phương án đều có một xe phải đợi máy 0 xong một xe 3 giây.
- Mức 3: 3 xe, đáp án `3`.

**Bài 3.**
- Mức 1: hai câu `if` độc lập (không `elif`).
- Mức 2: giá trị chờ là `a` khi chọn máy 0, `b` khi chọn máy 1.
- Mức 3: `out.append(((r0 - d, r1, d, max(0, b - a)), a))`.

**Bài 4.**
- Mức 1: giữ `served` = số lớp khác rỗng.
- Mức 2: khi gộp trạng thái trùng, giữ `min` của `max(w, chờ)`.
- Mức 3: nếu lớp mới rỗng thì `break`.

**Bài 5.**
- Mức 1: bỏ hẳn `a, b`; chỉ còn phân xăng.
- Mức 2: tập `{(r0, r1)}` như hàm `max_served` ở cách 4.
- Mức 3: chính là `max_served` — "được phép khi xe trước xong" không ảnh hưởng việc còn xăng hay không.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert simulate([10, 15], [12, 17], [0, 1]) == (2, 0, 0)

# Bài 2
assert Solution().minMaxWaitingTime([3, 3, 3], [6, 3]) == 3
assert max_served([3, 3, 3], [6, 3]) == 3

# Bài 3
def step_ans(state, d):
    r0, r1, a, b = state
    out = []
    if r0 >= d:
        out.append(((r0 - d, r1, d, max(0, b - a)), a))
    if r1 >= d:
        out.append(((r0, r1 - d, max(0, a - b), d), b))
    return out

assert step_ans((16, 13, 0, 0), 6) == [((10, 13, 6, 0), 0), ((16, 7, 0, 6), 0)]
assert step_ans((8, 8, 0, 0), 10) == []

# Bài 4
def my_solve_ans(demand, fuel):
    layer = {(fuel[0], fuel[1], 0, 0): 0}
    served = 0
    for i, d in enumerate(demand):
        nxt = {}
        for st, w in layer.items():
            for ns, wait in step_ans(st, d):
                val = max(w, wait)
                if ns not in nxt or val < nxt[ns]:
                    nxt[ns] = val
        if not nxt:
            break
        layer, served = nxt, i + 1
    return -1 if served == 0 else min(layer.values())

rng = random.Random(8)
for _ in range(300):
    n = rng.randint(1, 9)
    demand = [rng.randint(1, 8) for _ in range(n)]
    fuel = [rng.randint(1, 20), rng.randint(1, 20)]
    assert my_solve_ans(demand, fuel) == Solution().minMaxWaitingTime(demand, fuel)
print("Bài 1–4 OK")

Bài 1–4 OK


In [27]:
# Bài 5: chỉ cần phân xăng -> chính là max_served
def solve_finish_ans(demand, fuel):
    states = {(fuel[0], fuel[1])}
    k = 0
    for d in demand:
        nxt = set()
        for r0, r1 in states:
            if r0 >= d: nxt.add((r0 - d, r1))
            if r1 >= d: nxt.add((r0, r1 - d))
        if not nxt: break
        states, k = nxt, k + 1
    return k

def finish_bruteforce(demand, fuel):
    best = 0
    def dfs(i, r0, r1):
        nonlocal best
        best = max(best, i)
        if i == len(demand): return
        d = demand[i]
        if r0 >= d: dfs(i + 1, r0 - d, r1)
        if r1 >= d: dfs(i + 1, r0, r1 - d)
    dfs(0, fuel[0], fuel[1])
    return best

rng = random.Random(21)
for _ in range(300):
    n = rng.randint(1, 10)
    demand = [rng.randint(1, 8) for _ in range(n)]
    fuel = [rng.randint(1, 20), rng.randint(1, 20)]
    assert solve_finish_ans(demand, fuel) == finish_bruteforce(demand, fuel)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP trạng thái tương đối, mục tiêu hai tầng:**

```python
from functools import lru_cache

class Solution:
    def minMaxWaitingTime(self, demand, fuel):
        n = len(demand)

        @lru_cache(None)
        def go(i, r0, r1, a, b):          # a, b = còn bận bao lâu (so với bây giờ)
            if i == n:
                return (0, 0)
            d, best = demand[i], None
            if r0 >= d:                    # chọn máy 0: chờ a
                neg, w = go(i + 1, r0 - d, r1, d, max(0, b - a))
                best = (neg - 1, max(a, w))
            if r1 >= d:                    # chọn máy 1: chờ b
                neg, w = go(i + 1, r0, r1 - d, max(0, a - b), d)
                c = (neg - 1, max(b, w))
                best = c if best is None or c < best else best
            return best or (0, 0)

        neg, w = go(0, fuel[0], fuel[1], 0, 0)
        return -1 if neg == 0 else w
```

**Checklist trước khi submit:**

- [ ] Xe `i` được phép lúc xe `i-1` **bắt đầu** (không phải bơm xong).
- [ ] Trạng thái dùng thời gian **tương đối**, luôn kẹp `max(0, ·)`.
- [ ] Chỉ chọn máy có `rem >= demand[i]`; cả hai thiếu thì dừng.
- [ ] So sánh theo `(-số_xe, max_wait)`; phục vụ 0 xe trả `-1`.
- [ ] Xe không được phục vụ **không** tính vào thời gian chờ.
- [ ] Đã đối chiếu với brute force `simulate` trên input ngẫu nhiên nhỏ.